# 04 嵌入模型与检索链路体检

**测什么**: BGE 嵌入模型能否加载并产出 1024 维向量、法条检索(`fetch_laws`)、
案例检索(`get_retriever().search`)、重排序(CROSS-ENCODER)是否生效。

**判定**: 嵌入模型加载成功为 PASS; 检索链路要求库连通, 库不通时降级为 SKIP 并说明。
首次加载 BGE 约需 30-90 秒(CPU), 之后走单例。

**前置**: 01 册数据与模型缓存; 03 册的库连通性决定检索能否出结果。


In [ ]:
import asyncio, os, sys
from pathlib import Path

for cand in (Path.cwd(), *Path.cwd().parents):
    if (cand / "nbkit.py").is_file():
        NB_DIR = cand
        break
    if (cand / "tests_ipynb" / "nbkit.py").is_file():
        NB_DIR = cand / "tests_ipynb"
        break
else:
    raise RuntimeError("未找到 nbkit.py")

sys.path.insert(0, str(NB_DIR))

from nbkit import Checks, bootstrap

ROOT = bootstrap()
checks = Checks("04 检索体检")

print("解释器  :", sys.executable)
print("仓库根  :", ROOT)
print("HF_HOME :", os.getenv("HF_HOME", "(未设置)"))


In [ ]:
import time
from lawApp_LangGraph.config import settings as s

QUERY = "离婚时婚后购买的房产如何分割"
print("嵌入模型    :", s.memory_embed_model, f"dims={s.embed_dim}")
print("重排序模型  :", s.rerank_model, f"enabled={s.rerank_enabled}")
print("检索后端    :", os.getenv("RETRIEVER_BACKEND", "(未 load_dotenv → 默认 pgvector)"))

## 1. 嵌入模型加载与向量维度

In [ ]:
from lawApp_LangGraph.RAG_service.embedder import embed_query

t0 = time.time()
try:
    vec = await asyncio.wait_for(embed_query(QUERY), 300)
except Exception as e:
    checks.fail("BGE 嵌入可用", f"{type(e).__name__}: {str(e)[:200]}")
    vec = None
else:
    dt = time.time() - t0
    checks.expect(
        len(vec) == s.embed_dim,
        "BGE 嵌入可用",
        ok_detail=f"{dt:.1f}s(冷启动含模型加载) | dim={len(vec)}",
        fail_detail=f"维度不符: 期望 {s.embed_dim}, 实际 {len(vec)}",
    )

## 2. 重排序模型

In [ ]:
from lawApp_LangGraph.RAG_service.embedder import rerank

if s.rerank_enabled == "0":
    checks.skip("CrossEncoder 重排序", "RERANK_ENABLED=0 已禁用")
elif vec is None:
    checks.skip("CrossEncoder 重排序", "嵌入不可用, 先修上一项")
else:
    docs = [{"chunk_text": "婚后共同财产原则上均等分割"}, {"chunk_text": "婚前财产归个人所有"}]
    t0 = time.time()
    try:
        scores = await asyncio.wait_for(rerank(QUERY, docs), 300)
    except Exception as e:
        checks.fail("CrossEncoder 重排序", f"{type(e).__name__}: {str(e)[:160]}")
    else:
        checks.expect(
            scores is not None and len(scores) == len(docs),
            "CrossEncoder 重排序",
            ok_detail=f"{time.time() - t0:.1f}s | scores={[round(x, 3) for x in (scores or [])]}",
            fail_detail="返回 None(模型加载失败或已禁用)",
        )

## 3. Postgres 前置

检索链路依赖 `law_vector` / `law_cases` 两张表。库不通时下面三项必然失败,
且每次要等 30 秒连接池超时, 故先探一次并把根因单列, 避免重复三次长等待。

In [ ]:
import psycopg
from lawApp_LangGraph.db import build_dsn

PG_HINT = "前置不满足: Postgres 未连通(根因见 03 册, 修 DB_PASSWORD / DATABASE_URL 后重跑)"
try:
    _conn = psycopg.connect(
        build_dsn().replace("postgresql+psycopg", "postgresql"), connect_timeout=3
    )
    _conn.close()
    PG_OK = True
    checks.ok("Postgres 前置", "已连通")
except Exception as e:
    PG_OK = False
    checks.fail("Postgres 前置", f"{type(e).__name__}: {str(e)[:120]} → {PG_HINT}")

## 4. 法条检索 `fetch_laws`(law_vector 表)

In [ ]:
from lawApp_LangGraph.tools.db_tools import fetch_laws

if not PG_OK:
    checks.skip("法条检索 fetch_laws", PG_HINT)
else:
    t0 = time.time()
    res = await asyncio.wait_for(fetch_laws.ainvoke({"query": QUERY, "top_k": 3}), 300)
    status = res.get("status")
    laws = res.get("law_results") or []
    if status == "error":
        checks.fail("法条检索 fetch_laws", f"status=error count={res.get('count')} (库层报错)")
    else:
        checks.expect(
            len(laws) > 0,
            "法条检索 fetch_laws",
            ok_detail=f"{time.time() - t0:.1f}s | {len(laws)} 条 | 首条《{laws[0].law_title}》第{laws[0].article_number}条",
            fail_detail="返回 0 条 → law_vector 表为空, 需入库脚本灌数据",
        )

## 5. 案例检索 `get_retriever().search`(law_cases 表 + 重排)

In [ ]:
from lawApp_LangGraph.RAG_service.base import get_retriever

if not PG_OK:
    checks.skip("案例检索 retriever.search", PG_HINT)
else:
    t0 = time.time()
    try:
        hits = await asyncio.wait_for(get_retriever().search(QUERY, top_k=10, rerank_top_n=3), 300)
    except Exception as e:
        checks.fail("案例检索 retriever.search", f"{type(e).__name__}: {str(e)[:200]}")
    else:
        checks.expect(
            len(hits) > 0,
            "案例检索 retriever.search",
            ok_detail=f"{time.time() - t0:.1f}s | {len(hits)} 条 | top_score={hits[0]['hybrid_score']:.3f}",
            fail_detail="返回 0 条 → law_cases 表为空, 需入库脚本灌数据",
        )

## 6. RAG 工具链(检索 → 评估)

In [ ]:
from lawApp_LangGraph.tools.rag_tools import evaluate_case_relevance, retrieve_legal_knowledge

if not PG_OK:
    checks.skip("RAG 工具链 retrieve → evaluate", PG_HINT)
else:
    res = await asyncio.wait_for(
        retrieve_legal_knowledge.ainvoke({"query": QUERY, "top_k": 10, "rerank_top_n": 3}), 300
    )
    docs = res.get("rag_documents") or []
    if res.get("status") == "error":
        checks.fail("RAG 工具链 retrieve", f"{str(res.get('message', ''))[:160]}")
    elif not docs:
        checks.skip("RAG 工具链 retrieve → evaluate", "检索为空(案例库无数据)")
    else:
        ev = evaluate_case_relevance.invoke({"documents": docs}).get("evaluation")
        checks.ok(
            "RAG 工具链 retrieve → evaluate",
            f"{len(docs)} 条 → 三档: 高{ev.correct_count}/中{ev.ambiguous_count}/低{ev.incorrect_count} | 结论={ev.quality_verdict}",
        )

## 汇总

In [ ]:
print(checks.report())